# 03 Train NAND Best
Best-Path Training (InfoNCE, 818->1024->256).

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
DEVICE = "auto"


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
import numpy as np
import pandas as pd

from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.common.io_schema import read_parquet
from src.approaches.nand.train import train_nand_across_seeds

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "03_run_consistency.json",
    extras={"notebook": "03_train_nand_best", "latest_context_path": str(latest_context_path)},
)

training_cfg = MODEL["training"]
subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
ckpt_dir = RUN_DIRS["checkpoints"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
lspo_pairs = read_parquet(subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet")
lspo_chars = np.load(emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy")
lspo_text = np.load(emb_dir / f"lspo_specter_{RUN_STAGE}.npy")

print("RUN_ID:", RUN_ID)
print("Model:", MODEL.get("name"))
print("Loss:", training_cfg.get("loss"))
print("Architecture:", f"{training_cfg['input_dim']} -> {training_cfg['hidden_dim']} -> {training_cfg['output_dim']}")


RUN_ID: smoke_20260212T200818Z_6b207060
Model: nand_best
Loss: infonce
Architecture: 818 -> 1024 -> 256


In [3]:
default_seeds = training_cfg.get("seeds", [1,2,3,4,5])
seeds = [default_seeds[0]] if RUN_STAGE in {"smoke", "mini"} else default_seeds

manifest_path = RUN_DIRS["metrics"] / "03_train_manifest.json"
manifest = train_nand_across_seeds(
    mentions=lspo_mentions,
    pairs=lspo_pairs,
    chars2vec=lspo_chars,
    text_emb=lspo_text,
    model_config=training_cfg,
    seeds=seeds,
    run_id=RUN_ID,
    output_dir=ckpt_dir,
    metrics_output=manifest_path,
    device=DEVICE,
)
manifest


{'run_id': 'smoke_20260212T200818Z_6b207060',
 'runs': [{'seed': 1,
   'checkpoint': 'C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\checkpoints\\smoke_20260212T200818Z_6b207060\\smoke_20260212T200818Z_6b207060_seed1.pt',
   'threshold': -0.22699999999999998,
   'threshold_selection_status': 'ok',
   'threshold_source': 'val_f1_opt',
   'val_class_counts': {'pos': 35, 'neg': 4},
   'test_class_counts': {'pos': 33, 'neg': 3},
   'val_stats': {'f1': 0.9459459459459459,
    'precision': 0.8974358974358975,
    'recall': 1.0,
    'accuracy': 0.8974358974358975},
   'test_metrics': {'f1': 0.9565217391304348,
    'precision': 0.9166666666666666,
    'recall': 1.0,
    'accuracy': 0.9166666666666666}}],
 'best_seed': 1,
 'best_checkpoint': 'C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\checkpoints\\smoke_20260212T200818Z_6b207060\\smoke_20260212T200818Z_6b

In [4]:
rows = []
for r in manifest["runs"]:
    rows.append({
        "seed": r["seed"],
        "checkpoint": r["checkpoint"],
        "threshold": r["threshold"],
        "val_f1": r["val_stats"].get("f1"),
        "val_precision": r["val_stats"].get("precision"),
        "val_recall": r["val_stats"].get("recall"),
        "test_f1": r["test_metrics"].get("f1"),
        "test_precision": r["test_metrics"].get("precision"),
        "test_recall": r["test_metrics"].get("recall"),
    })

print("Seed summary")
display(pd.DataFrame(rows).sort_values("val_f1", ascending=False))

diag_rows = []
for r in manifest["runs"]:
    v = r.get("val_class_counts", {}) or {}
    t = r.get("test_class_counts", {}) or {}
    diag_rows.append({
        "seed": r["seed"],
        "threshold": r.get("threshold"),
        "threshold_selection_status": r.get("threshold_selection_status"),
        "threshold_source": r.get("threshold_source"),
        "val_pos": int(v.get("pos", 0)),
        "val_neg": int(v.get("neg", 0)),
        "test_pos": int(t.get("pos", 0)),
        "test_neg": int(t.get("neg", 0)),
    })

print("Threshold diagnostics")
display(pd.DataFrame(diag_rows))


Seed summary


,seed,checkpoint,threshold,val_f1,val_precision,val_recall,test_f1,test_precision,test_recall
0,1,C:\Users\rapha\Documents\Studium\Promotionsstu...,-0.227,0.945946,0.897436,1.0,0.956522,0.916667,1.0


Threshold diagnostics


,seed,threshold,threshold_selection_status,threshold_source,val_pos,val_neg,test_pos,test_neg
0,1,-0.227,ok,val_f1_opt,35,4,33,3
